# Pipeline Cálculo Vectorial — Exploración 2026-1

Notebook principal para procesar notas del ciclo 2026-1.  
Ejecutar las celdas en orden. Las celdas marcadas con ⚙️ requieren configuración manual.

**Orden de ejecución:**
1. Imports y configuración
2. Carga del dashboard base (Notas)
3. Carga de Canvas Teoría 1 y 2
4. Carga de Gradescope EAs
5. Merge de todas las fuentes
6. Ejecución de cálculos
7. Visualización de resultados
8. Exportar CSV de salida
9. (Opcional) Actualizar Google Sheets

## 1. Imports y configuración de paths

In [ ]:
import sys
import yaml
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Agregar el directorio raíz al path para importar src/
RAIZ = Path().resolve().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.ingesta import load_canvas_csv, load_gradescope_csv, load_gradescope_ea
from src.merge import merge_con_fallback, merge_todas_fuentes, reporte_merge
from src.calculos import ejecutar_calculos
from src.reporte import (
    estadisticas_por_seccion,
    alumnos_en_riesgo,
    resumen_curso,
)

# Configuración de paths
CICLO = "2026-1"
DIR_DATA = RAIZ / "data" / CICLO
DIR_CANVAS = DIR_DATA / "raw" / "canvas"
DIR_GS = DIR_DATA / "raw" / "gradescope"
DIR_EA = DIR_GS / "EA"
DIR_TAREAS = DIR_GS / "tareas"
DIR_PROCESADO = DIR_DATA / "processed"
DIR_OUTPUT = DIR_DATA / "output"

# Cargar configuración del ciclo
with open(RAIZ / "config" / f"{CICLO}.yaml") as f:
    CONFIG = yaml.safe_load(f)

print(f"Ciclo: {CONFIG['ciclo']}")
print(f"Directorio de datos: {DIR_DATA}")
print(f"Secciones auditorio: {CONFIG['secciones']['auditorio']}")
print(f"Secciones aula: {CONFIG['secciones']['aula']}")

## 2. Carga del dashboard base (Notas)

El dashboard base contiene la lista oficial de alumnos matriculados,  
descargado desde el sistema EDU de UTEC o el Google Sheet 'Notas'.

In [ ]:
# ⚙️ Ajustar el nombre del archivo según el que tengas en data/2026-1/raw/
ARCHIVO_BASE = DIR_DATA / "raw" / "Notas.csv"

df_base = pd.read_csv(
    ARCHIVO_BASE,
    sep=";",
    encoding="latin-1",
    dtype=str,
)

# Limpiar filas completamente vacías
df_base = df_base.dropna(how="all").reset_index(drop=True)

# Asegurarse de que Código sea string para el merge
if "Código" in df_base.columns:
    df_base["Código"] = df_base["Código"].str.strip()

if "Correo" in df_base.columns:
    df_base["Correo"] = df_base["Correo"].str.strip()

print(f"Dashboard base cargado: {len(df_base)} alumnos")
print(f"Columnas: {list(df_base.columns)}")
df_base.head()

## 3. Carga de Canvas Teoría 1 y 2

Canvas exporta los CSVs con:
- Fila 2 = 'Points Possible' (se elimina automáticamente)
- Decimales con coma: '20,00' (se convierten automáticamente)
- Encoding puede ser Latin-1 o UTF-8 (se detecta automáticamente)

In [ ]:
# ⚙️ Ajustar nombres de archivo según los que tengas descargados de Canvas
ARCHIVO_CANVAS1 = DIR_CANVAS / CONFIG["fuentes"]["canvas"]["teoria_1"]
ARCHIVO_CANVAS2 = DIR_CANVAS / CONFIG["fuentes"]["canvas"]["teoria_2"]

# Cargar Canvas Teoría 1 (secciones aula)
if ARCHIVO_CANVAS1.exists():
    df_canvas1 = load_canvas_csv(ARCHIVO_CANVAS1, CONFIG)
    print(f"Canvas Teoría 1 cargado: {len(df_canvas1)} filas")
    print(f"Columnas disponibles: {list(df_canvas1.columns)}")
else:
    print(f"⚠️ No se encontró: {ARCHIVO_CANVAS1}")
    df_canvas1 = pd.DataFrame()

# Cargar Canvas Teoría 2 (secciones auditorio)
if ARCHIVO_CANVAS2.exists():
    df_canvas2 = load_canvas_csv(ARCHIVO_CANVAS2, CONFIG)
    print(f"\nCanvas Teoría 2 cargado: {len(df_canvas2)} filas")
    print(f"Columnas disponibles: {list(df_canvas2.columns)}")
else:
    print(f"⚠️ No se encontró: {ARCHIVO_CANVAS2}")
    df_canvas2 = pd.DataFrame()

# Vista previa
if not df_canvas1.empty:
    df_canvas1[["Código", "Correo", "Sección"]].head()

In [ ]:
# Combinar Canvas 1 y 2 en un solo DataFrame
# ⚙️ Ajustar los nombres de columnas según el CSV real de Canvas

# Renombrar columnas de evaluación según la configuración
# Ejemplo: la columna 'Tarea 1' de Canvas → columna 'T1' en el pipeline
mapeo_canvas = {
    CONFIG["evaluaciones"]["T1"].get("columna", "Tarea 1"): "T1",
    CONFIG["evaluaciones"]["T3A"].get("columna", "Tarea 3 - Parte A"): "T3A",
    CONFIG["evaluaciones"]["T4"].get("columna", "Tarea 4"): "T4",
}

partes_canvas = []
for df_c in [df_canvas1, df_canvas2]:
    if not df_c.empty:
        df_ren = df_c.rename(columns=mapeo_canvas)
        cols_utiles = ["Código", "Correo", "Sección"] + [
            c for c in ["T1", "T3A", "T4"] if c in df_ren.columns
        ]
        partes_canvas.append(df_ren[cols_utiles])

if partes_canvas:
    df_canvas_all = pd.concat(partes_canvas, ignore_index=True)
    df_canvas_all = df_canvas_all.drop_duplicates(subset=["Código"], keep="first")
    print(f"Canvas combinado: {len(df_canvas_all)} alumnos")
    df_canvas_all.head()
else:
    df_canvas_all = pd.DataFrame(columns=["Código", "Correo", "Sección"])
    print("⚠️ No hay datos de Canvas disponibles.")

## 4. Carga de Gradescope EAs

Gradescope genera un CSV por sección. La función `load_gradescope_ea`
los concatena automáticamente para todas las secciones del ciclo.

In [ ]:
# Todas las secciones del ciclo
TODAS_LAS_SECCIONES = (
    CONFIG["secciones"]["auditorio"] + CONFIG["secciones"]["aula"]
)
PATRON_EA = CONFIG["fuentes"]["gradescope"]["ea_patron"]

# Cargar EAs disponibles
dfs_ea = {}
for n_ea in [1, 2, 3]:  # Agregar 4,5,6 cuando estén disponibles
    df_ea = load_gradescope_ea(
        carpeta=DIR_EA,
        n_ea=n_ea,
        secciones=TODAS_LAS_SECCIONES,
        patron=PATRON_EA,
    )
    if not df_ea.empty:
        dfs_ea[f"EA{n_ea}"] = df_ea
        print(f"EA{n_ea}: {len(df_ea)} alumnos cargados")
    else:
        print(f"EA{n_ea}: sin datos disponibles")

In [ ]:
# Cargar Tarea 2 desde Gradescope
ARCHIVO_T2 = DIR_TAREAS / CONFIG["fuentes"]["gradescope"]["tarea_2"]
if ARCHIVO_T2.exists():
    df_t2 = load_gradescope_csv(ARCHIVO_T2)
    df_t2 = df_t2.rename(columns={"Total Score": "T2"})
    print(f"Tarea 2 cargada: {len(df_t2)} alumnos")
else:
    print(f"⚠️ No se encontró Tarea 2: {ARCHIVO_T2}")
    df_t2 = pd.DataFrame()

# Cargar Tarea 3B desde Gradescope
ARCHIVO_T3B = DIR_TAREAS / CONFIG["fuentes"]["gradescope"]["tarea_3b"]
if ARCHIVO_T3B.exists():
    df_t3b = load_gradescope_csv(ARCHIVO_T3B)
    df_t3b = df_t3b.rename(columns={"Total Score": "T3B"})
    print(f"Tarea 3B cargada: {len(df_t3b)} alumnos")
else:
    print(f"⚠️ No se encontró Tarea 3B: {ARCHIVO_T3B}")
    df_t3b = pd.DataFrame()

## 5. Merge de todas las fuentes

Usamos el Código universitario como llave primaria.  
Para alumnos con DNI o código especial en Canvas, el fallback es el Correo.

In [ ]:
# Armar el diccionario de fuentes para merge
fuentes = {}

# Canvas
if not df_canvas_all.empty:
    fuentes["canvas"] = {
        "df": df_canvas_all,
        "columnas": [c for c in ["T1", "T3A", "T4", "Sección"] if c in df_canvas_all.columns],
        "sufijo": "_canvas",
    }

# Tarea 2 (Gradescope)
if not df_t2.empty:
    fuentes["tarea2"] = {"df": df_t2, "columnas": ["T2"], "sufijo": "_gs"}

# Tarea 3B (Gradescope)
if not df_t3b.empty:
    fuentes["tarea3b"] = {"df": df_t3b, "columnas": ["T3B"], "sufijo": "_gs"}

# EAs
for nombre_ea, df_ea in dfs_ea.items():
    fuentes[nombre_ea] = {
        "df": df_ea,
        "columnas": [nombre_ea],
        "sufijo": "_gs",
    }

# Ejecutar todos los merges
df_merged = merge_todas_fuentes(df_base, fuentes, verbose=True)

# Reporte de cobertura
reporte = reporte_merge(df_merged)
df_merged.shape

## 6. Ejecución de cálculos

Calcula PT1, PT2, PEA1, PEA2, BPEA1_prov, BPEA2, PfEA1, PfEA2, TA1, TA2, EP, EF, NF.

In [ ]:
# ⚙️ Ajustar las columnas de videos de Actividades Previas si ya están disponibles
# Ejemplo: las columnas se llaman 'AP1V1', 'AP1V2', ..., 'AP6V2' en el DataFrame

COLS_AP_BPEA1 = [
    "AP1V1", "AP1V2",
    "AP2V1", "AP2V2",
    "AP3V1", "AP3V2",
    "AP4V1", "AP4V2",
    "AP5V1", "AP5V2",
    "AP6V1", "AP6V2",
]

# Semana 9 tiene 2 videos (AP7V1, AP7V2), resto 1 video cada una
COLS_AP_BPEA2 = ["AP7V1", "AP7V2", "AP8V1", "AP9V1", "AP10V1"]

df_calculado = ejecutar_calculos(
    df_merged,
    CONFIG,
    cols_ap_bpea1=COLS_AP_BPEA1,
    cols_ap_bpea2=COLS_AP_BPEA2,
)

print(f"\nDataFrame calculado: {df_calculado.shape}")

## 7. Visualización de resultados por sección

In [ ]:
# Estadísticas globales del curso
df_resumen = resumen_curso(df_calculado)

In [ ]:
# Estadísticas por sección para NF (o cualquier columna calculada)
if "NF" in df_calculado.columns:
    df_stats_seccion = estadisticas_por_seccion(df_calculado, "NF")
    print("\nEstadísticas de NF por sección:")
    display(df_stats_seccion)

    # Gráfica de distribución de NF por sección
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Boxplot por sección
    df_plot = df_calculado[["Sección", "NF"]].dropna()
    if not df_plot.empty:
        df_plot["Sección"] = df_plot["Sección"].astype(str)
        sns.boxplot(data=df_plot, x="Sección", y="NF", ax=axes[0], palette="muted")
        axes[0].set_title("Distribución de NF por Sección")
        axes[0].set_ylabel("Nota Final")
        axes[0].axhline(10.5, color="red", linestyle="--", label="Mín. aprobatorio")
        axes[0].legend()

    # Histograma de NF global
    df_calculado["NF"].dropna().hist(
        bins=20, ax=axes[1], color="steelblue", edgecolor="white"
    )
    axes[1].axvline(10.5, color="red", linestyle="--", label="Mín. aprobatorio")
    axes[1].set_title("Distribución Global de NF")
    axes[1].set_xlabel("Nota Final")
    axes[1].set_ylabel("Número de alumnos")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(DIR_OUTPUT / "distribucion_NF.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Alumnos en riesgo (NF proyectada < 10.5)
df_riesgo = alumnos_en_riesgo(df_calculado, umbral=10.5)
if not df_riesgo.empty:
    display(df_riesgo)

## 8. Exportar CSV de salida

Guarda el DataFrame final como respaldo local en `data/2026-1/output/`.

In [ ]:
DIR_OUTPUT.mkdir(parents=True, exist_ok=True)

ARCHIVO_SALIDA = DIR_OUTPUT / "notas_calculadas_2026-1.csv"
df_calculado.to_csv(ARCHIVO_SALIDA, index=False, encoding="utf-8")
print(f"✅ Notas exportadas: {ARCHIVO_SALIDA}")
print(f"   Filas: {len(df_calculado)}, Columnas: {len(df_calculado.columns)}")

## 9. (Opcional) Actualizar Google Sheets

⚙️ Requiere:
- Archivo `credentials/service_account.json`
- Sheet ID configurado en `config/2026-1.yaml`
- Sheet compartido con el email de la Service Account

In [ ]:
# Descomentar para ejecutar la actualización de Google Sheets

# from src.gdrive import conectar_gspread, escribir_columnas_notas, escribir_formativa
#
# CREDENTIALS = RAIZ / "credentials" / "service_account.json"
# SHEET_ID = CONFIG["dashboard"]["sheet_id"]  # ⚙️ Reemplazar en config/2026-1.yaml
#
# gc = conectar_gspread(CREDENTIALS)
#
# # Columnas a escribir en la hoja Notas
# columnas_a_escribir = [
#     "EA1", "EA2", "EA3",
#     "PT1", "PEA1", "BPEA1_prov", "PfEA1",
#     "TA1", "EP", "EF", "NF",
# ]
#
# escribir_columnas_notas(
#     sheet_id=SHEET_ID,
#     df=df_calculado,
#     columnas=columnas_a_escribir,
#     gc=gc,
# )
#
# # Actualizar hoja Formativa con bonificaciones
# escribir_formativa(
#     sheet_id=SHEET_ID,
#     df_formativa=df_calculado,
#     gc=gc,
# )
print("(Celda de actualización de Google Sheets — descomentar para usar)")